<a href="https://colab.research.google.com/github/ALS240/RAG_chatboat/blob/main/Employee_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 10.6 MB/s eta 0:00:00


In [3]:
from transformers import pipeline

generator = pipeline("text-generation", model="google/flan-t5-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [4]:
#data
documents = [
    ("working hours office timing",
     "The working hours are from 9:30 AM to 6:30 PM, totaling 9 hours per day for 5 days a week."),

    ("leaves paid leaves policy",
     "Employees are entitled to 20 paid leaves per year as part of the company policy."),

    ("work from home wfh policy",
     "Employees are allowed to work from home for up to 2 days per month depending on approval."),

    ("salary payment date",
     "Salaries are credited on the last working day of each month."),

    ("internship availability",
     "Yes, the company offers internship opportunities for students and freshers."),

    ("attendance late login policy",
     "Employees are allowed late login up to 3 times per month under the attendance policy."),

    ("notice period duration",
     "The standard notice period for employees is 30 days before leaving the company.")
]

In [5]:
#Split Q & A
from sentence_transformers import SentenceTransformer
questions = [q for q, a in documents]
answers = [a for q, a in documents]
#Embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(questions)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
#FAISS
import faiss
import numpy as np

index = faiss.IndexFlatL2(len(doc_embeddings[0]))
index.add(np.array(doc_embeddings))

In [7]:
#Normalize query
def normalize_query(query):
    query = query.lower()

    replacements = {
        "wfh": "work from home",
        "workfromhome": "work from home",
        "working time": "working hours",
        "office time": "working hours"
    }

    for k, v in replacements.items():
        query = query.replace(k, v)

    return query

In [8]:
#Small talk
def handle_small_talk(query):
    responses = ["ok", "okay", "thanks", "thank you", "tq", "done"]

    if query.lower().strip() in responses:
        return "Thank you, you're welcome!"

    return None

In [9]:
#Retrieval
def retrieve_context(query, k=1):
    query = normalize_query(query)

    query_vec = embed_model.encode([query])
    D, I = index.search(np.array(query_vec), k)

    score = D[0][0]
    idx = I[0][0]

    if idx >= len(answers):
        return None, score

    return answers[idx], score

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
def generate_answer(query, context):
    if not context:
        context = "No relevant company policy found."

    prompt = f"""
    You are an HR assistant.

    STRICT RULES:
    - Use ONLY the given context
    - Do NOT make up information
    - If context is not relevant, say you don't know

    Context: {context}
    Question: {query}

    Answer clearly:
    """

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=100)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [12]:
#Fallback
def fallback():
    return "Please contact mail SupportSW@gmail.com for further information."

In [13]:
#Chatbot
def chatbot(query):
    small = handle_small_talk(query)
    if small:
        return small

    answer, score = retrieve_context(query)

    if answer and score < 0.8:
        return answer

    return generate_answer(query, answer)

In [14]:
!pip install fastapi uvicorn nest-asyncio pyngrok

In [15]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

# Request format
class QueryRequest(BaseModel):
    query: str

# API endpoint
@app.post("/chat")
def chat_api(request: QueryRequest):
    try:
        print("User Query:", request.query)

        response = chatbot(request.query)

        print("Response:", response)

        return {"answer": response}

    except Exception as e:
        print("ERROR:", str(e))
        return {"error": str(e)}

In [16]:
!ngrok config add-authtoken 3Bpr4DJtY7L38GfocJF97SnhO6z_2ZUXoFTpssPHGFTn8AewT

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [17]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from threading import Thread

nest_asyncio.apply()

#  Kill old tunnels
ngrok.kill()

# Start new tunnel
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

# Run server
def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run_uvicorn)
thread.start()

Public URL: NgrokTunnel: "https://cespitose-kourtney-stellularly.ngrok-free.dev" -> "http://localhost:8000"
